# Simple Single Lens (with added Complexities) simulation with MeepSAT

In this Tutorial we are going to simulate the different baffles (in time-reverse) that we included in the previous tutorial.

`Note`: Some of the simulations in this 2nd Tutorial might take a bit of time to run due to system size and resolution, so if you want to test/debug things you can comment out the `sim.check_resolution_and_pml()` and run at a lower resolution! But for accurate results, please use the recommended resolution returned from the `sim.check_resolution_and_pml()` module!

In [ ]:
# Importing various Python libraries and MeepSAT modules
import sys
import os
import site
from pathlib import Path
import meep as mp
import numpy as np
import h5py
import matplotlib.pyplot as plt
import time
import json
import math

# Importing the MEEPSAT librarires
import meepsat.simulator as sim
import meepsat.meep_geometry as comp_meep
import meepsat.permittivity_components as comp_eps
import meepsat.stepfunctions as stepfunctions
import meepsat.json_to_script as json_to_script
import meepsat.field_analysis as mpsat_analysis
import meepsat.helpers as mpsat_helpers

def run_simulation(data, source_freq, res, savepath_dir, runtime, beam_waist,
                   forebaffle_scaling_factor, forebaffle_spline_degree, forebaffle_spline_smoothing,
                   forebaffle_height=50, forebaffle_base=86.60*2, forebaffle_thickness=6,
                   forebaffle_absorber_thick=3, forebaffle_num_periods=1, 
                   forebaffle_amplitude=0, absorber_epsilon_r=5.4, absorber_epsilon_i=0.8, add_flair=True,
                   flair_angle= None, flair_hypotenuse=None, flair_num_periods=1, flair_amplitude=0):
    """
    Run a MEEPSAT simulation with configurable forebaffle parameters.
    
    Args:
        data: Simulation configuration dictionary
        source_freq: Source frequency
        res: Resolution
        savepath_dir: Output directory path
        runtime: Simulation runtime
        beam_waist: Beam waist width in mm
        forebaffle_height: Height of forebaffle in mm
        forebaffle_base: Base of forebaffle in mm
        forebaffle_thickness: Thickness of forebaffle in mm
        forebaffle_absorber_thick: Absorber thickness in mm
        forebaffle_num_periods: Number of periods for forebaffle shape
        forebaffle_amplitude: Amplitude for forebaffle shape modulation
        absorber_epsilon_r: Real part of absorber permittivity
        absorber_epsilon_i: Imaginary part of absorber permittivity
    
    Returns:
        simulation: Completed MEEP simulation object
        savepath: Path where results are saved
    """


    # Update parameters in data
    data["sources"]['source1']["frequecy"] = float(source_freq)
    data["simulation"]['primary_params']['resolution'] = int(res)
    data["output"]["savepath"]["path"] = str(Path(savepath_dir)) + "/"
    data["sources"]['source1']["extra_args"]["width"] = float(beam_waist)

    savepath = data["output"]["savepath"]["path"]
    os.makedirs(savepath, exist_ok=True)
    print('Output directory path:', savepath)
    
    # Initialize MEEPSAT Simulation
    cell_X = data["simulation"]['primary_params']['cell_size']['x']
    cell_Y = data["simulation"]['primary_params']['cell_size']['y']
    cell_Z = data["simulation"]['primary_params']['cell_size']['z']
    
    mpsat_sim = sim.sim_init(
        sim_name=str(data["simulation"]["name"]),
        cell_size=[cell_X, cell_Y, cell_Z],
        smallest_freq=data["simulation"]['primary_params']['smallest_freq'],
        resolution=data["simulation"]['primary_params']['resolution'],
        boundary_layer_type=data['boundary_layers']['boundary']['type'],
        boundary_layer_size=data['boundary_layers']['boundary']['size'],
        factor_dpml=data['boundary_layers']['boundary']['factor_dpml'])
    
    # # Check resolution and PML thickness
    data, mpsat_sim = sim.check_resolution_and_pml(
        data=data,
        mpsat_sim=mpsat_sim,
        smallest_freq=data["simulation"]['primary_params']['smallest_freq'],
        highest_n=np.sqrt(5.4))
    
    mpsat_sim.print_simulation_parameters()
    
    # # Add sources
    # source_list = []
    # exec(json_to_script.source_script(data))
    
    # Add sources
    source_list = []
    local_vars = {
        'source_list': source_list,
        'data': data,
        'mp': mp,
        'mpsat_sim': mpsat_sim,
        'comp_meep': comp_meep,
        'np': np,
        'json_to_script': json_to_script
    }
    exec(json_to_script.source_script(data), globals(), local_vars)
    source_list = local_vars['source_list']
    print(f"Number of sources created: {len(source_list)}")
    if len(source_list) > 0:
        print(f"First source: {source_list[0]}")
    else:
        raise ValueError("WARNING: No sources were created!")
    
    # Add boundary layers
    x_left_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.X, side=mp.Low)
    x_right_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.X, side=mp.High)
    y_down_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.Y, side=mp.Low)
    y_up_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.Y, side=mp.High)
    
    custom_boundary_layers = [x_left_boundary, x_right_boundary, y_down_boundary, y_up_boundary]
    
    # Create epsilon map
    size_x, size_y, size_z = mpsat_sim.cell_size[0], mpsat_sim.cell_size[1], mpsat_sim.cell_size[2]
    res_int = int(mpsat_sim.resolution)
    epsilon_map = np.ones((int((size_x)*res_int+1), int((size_y)*res_int+1)), dtype=float)
    
    # Add absorbers (down and up)
    absorbers_down = comp_meep.Absorbers(
        p=6, p_h_ratio=1.5, taper_type='Linear',
        grid_size_sx=size_x, grid_size_sy=size_y, resolution=res_int,
        eps_array=epsilon_map, geometry_objects=[],
        z0=1, z1=1/math.sqrt(5.4),
        center_x_mm=0, center_y_mm=0, orientation="+y", angle_axis="x",
        substrate_thickness=[5, 53+50],
        substrate_material=[None, mp.perfect_electric_conductor],
        add_substrate=True,
        epsilon_r=5.4, epsilon_i=0.8,
        material_type='narrow_bandwidth_absorption',
        freq=data["sources"]['source1']["frequecy"],
        start_point=(-51, -40), end_point=(250, -40),
        overall_factor=1, plot_alpha=False, plot_profile=False,
        savepath=savepath, 
        plot_mesh=False)
    
    absorbers_up = comp_meep.Absorbers(
        p=6, p_h_ratio=1.5, taper_type='Linear',
        grid_size_sx=size_x, grid_size_sy=size_y, resolution=res_int,
        eps_array=epsilon_map, geometry_objects=[],
        z0=1, z1=1/math.sqrt(5.4),
        center_x_mm=0, center_y_mm=0, orientation="-y", angle_axis="x",
        substrate_thickness=[5, 53+50],
        substrate_material=[None, mp.perfect_electric_conductor],
        add_substrate=True,
        epsilon_r=5.4, epsilon_i=0.8,
        material_type='narrow_bandwidth_absorption',
        freq=data["sources"]['source1']["frequecy"],
        start_point=(-51, 40), end_point=(250, 40),
        overall_factor=1, plot_alpha=False, plot_profile=False,
        savepath=savepath, 
        plot_mesh=False)
    
    mpsat_sim.meep_geometry.extend(absorbers_up.assemble())
    mpsat_sim.meep_geometry.extend(absorbers_down.assemble())
    
    # Add lens, slab, and aperture
    exec(json_to_script.add_lens(data))
    exec(json_to_script.add_slab(data))
    exec(json_to_script.add_aperture(data))
    
    # Calculate forebaffle geometry parameters
    forebaffle_hypotenuse = np.sqrt(forebaffle_height**2 + forebaffle_base**2)
    forebaffle_angle = np.degrees(np.arctan(forebaffle_height / forebaffle_base))
    
    print(f"Forebaffle Geometry Parameters:")
    print(f"Height: {forebaffle_height} mm")
    print(f"Base: {forebaffle_base} mm")
    print(f"Hypotenuse: {forebaffle_hypotenuse:.2f} mm")
    print(f"Angle: {forebaffle_angle:.2f} degrees")
    
    # Bottom Forebaffle
    x_start_bottom, y_start_bottom = -48.4+3, -41-5
    
    fb_bottom = comp_meep.Forebaffle(
        mpsat_sim=mpsat_sim,
        epsilon_map=epsilon_map,
        angle_degrees=180+forebaffle_angle,
        x_vertex=x_start_bottom,
        y_vertex=y_start_bottom,
        hypotenuse=forebaffle_hypotenuse,
        material=mp.perfect_electric_conductor,
        name="Bottom Forebaffle",
        shape='spline',
        num_periods=forebaffle_num_periods,
        amplitude=forebaffle_amplitude,
        no_of_points=500,
        scaling_factor=3,
        spline_degree=3,
        spline_smoothing=1,
        fb_thickness=forebaffle_thickness,
        add_absorber=True,
        absorber_side='above',
        absorber_epsilon_real=absorber_epsilon_r,
        absorber_epsilon_imag=absorber_epsilon_i,
        absorber_thickness=forebaffle_absorber_thick)
    
    mpsat_sim.meep_geometry.extend(fb_bottom.assemble())
    
    # Top Forebaffle
    x_start_top, y_start_top = -48.4+3, 41+5
    
    fb_top = comp_meep.Forebaffle(
        mpsat_sim=mpsat_sim,
        epsilon_map=epsilon_map,
        angle_degrees=180-forebaffle_angle,
        x_vertex=x_start_top,
        y_vertex=y_start_top,
        hypotenuse=forebaffle_hypotenuse,
        material=mp.perfect_electric_conductor,
        name="Top Forebaffle",
        shape='spline',
        num_periods=forebaffle_num_periods,
        amplitude=forebaffle_amplitude,
        no_of_points=500,
        scaling_factor=3,
        spline_degree=3,
        spline_smoothing=1,
        fb_thickness=forebaffle_thickness,
        add_absorber=True,
        absorber_side='below',
        absorber_epsilon_real=absorber_epsilon_r,
        absorber_epsilon_imag=absorber_epsilon_i,
        absorber_thickness=forebaffle_absorber_thick)
    
    mpsat_sim.meep_geometry.extend(fb_top.assemble())
    
    # Define simulation object
    symmetries = [mp.Mirror(mp.Y, phase=+1)]
    
    simulation = mp.Simulation(
        cell_size=mpsat_sim.cell,
        sources=source_list,
        resolution=mpsat_sim.resolution,
        boundary_layers=custom_boundary_layers,
        geometry=mpsat_sim.meep_geometry,
        epsilon_input_file=data["output"]["savepath"]["path"] + data["output"]["epsilon_h5_file"]["filename"] + "_epsilon_map" + ".h5",
        # symmetries=symmetries,
        force_complex_fields=True)
    
    simulation.use_output_directory(savepath)
    
    import matplotlib.pyplot as plt
    plt.style.use('default')
    # Plot and save epsilon
    sim.plot_and_save_epsilon(
        simulation=simulation,
        savepath=savepath,
        filename_prefix="geometry_plot",
        epsilon_data_name="epsilon",
        size_x=size_x,
        size_y=size_y,
        vmin=0.5,
        vmax=3,
        cmap='viridis',
        figsize=(8, 4),
        dpi=300)
    
    # Set stepfunctions parameters
    stepfunctions.set_animation_params(anim_params={
        'image_every': data["output"]["animation_options"]["image_every"],
        'Nfps': data["output"]["animation_options"]["Nfps"],
        'anim_file_name': savepath + "/" + data["output"]["animation_options"]["movie_name"] + ".mp4"})
    
    stepfunctions.set_field_params(field_params={
        'size_x': size_x,
        'size_y': size_y,
        'savepath': savepath,
        'downsampling_factor_x': data["output"]["animation_options"]["downsample_x"],
        'downsampling_factor_y': data["output"]["animation_options"]["downsample_y"]})
    
    
    runtime_params = sim.calculate_runtime_parameters(
        source_freq=float(data["sources"]["source1"]["frequecy"]),
        resolution= mpsat_sim.resolution,
        steady_state_time = runtime,
        courant=simulation.Courant,
        min_periods_for_steady_state=10,
        periods_to_average=4,
        points_per_period=10,
        animation_timestep=data["output"]["animation_options"]["image_every"])
    
    # Run simulation
    simulation.run(
        mp.at_every(runtime_params["animation_timestep"], stepfunctions.Ez2_dB),
        mp.after_time(runtime_params["t0"], mp.at_every(runtime_params["dt"], stepfunctions.accumulate_efield_and_hfield)),
        mp.at_end(stepfunctions.save_animation),
        mp.at_end(stepfunctions.save_accumulated_fields),
        mp.at_end(stepfunctions.extract_xyzw),
        until=runtime_params["total_time"])
    
    print("Simulation completed.")
    
    # Save final JSON data
    with open(data["output"]["savepath"]["path"] + data["simulation"]["name"] + "_simulation_data.json", "w") as f:
        json.dump(data, f, indent=2)
    print(f"Simulation parameters saved to: {data['output']['savepath']['path']}{data['simulation']['name']}_simulation_data.json")
        

In [ ]:
json_file_path = 'auxilary_data/02_simple_single_lens_AddedComplexities_b/TRV/simple_single_lens_AddedComplexities_TRV_b.json'
data = mpsat_helpers.read_json(json_file_path)

# Some universal constants
c_mm_s = 299792458.0 * 1000.0  # Speed of light in mm/s (m/s -> mm/s)
# Frequency of the simulation
freq = 150.0  # Frequency in GHz
a = 1  # 1 meep unit = 1 mm  
wvl = c_mm_s / (freq * 1e9)  # Wavelength in mm
freq_meep = 1.0 / (wvl * a)
print("freq (meep units):", freq_meep)
freq = freq_meep

# Resolution
res = 4
# runtime
runtime = 600
# beam waist
beam_waist = 1.45756

# Savepath
savepath_dir = f'auxilary_data/02_simple_single_lens_AddedComplexities_b/TRV/noflairs/output_files/{freq}GHz'

# Define multiple forebaffle configurations
forebaffle_configs = [
    # Spline config_0
    {"forebaffle_scaling_factor": 3, "forebaffle_spline_degree": 3, "forebaffle_num_periods": 2, "forebaffle_amplitude": 2, "forebaffle_spline_smoothing": 1},
    # Curved config_1
    {"forebaffle_scaling_factor": 3, "forebaffle_spline_degree": 3, "forebaffle_num_periods": 0.4, "forebaffle_amplitude": 5, "forebaffle_spline_smoothing": 1},
    # Linear config_2
    {"forebaffle_scaling_factor": 3, "forebaffle_spline_degree": 3, "forebaffle_num_periods": 1, "forebaffle_amplitude": 0, "forebaffle_spline_smoothing": 1},
]

# Run simulations
for i, config in enumerate(forebaffle_configs):
    print(f"\n{'='*50}")
    print(f"Running simulation {i+1}/{len(forebaffle_configs)}")
    print(f"{'='*50}\n")
    
    # Create a copy of data for each simulation
    sim_data = json.loads(json.dumps(data))
    sim_data["simulation"]["name"] += f"_config_{i}"
    
    # Edit the beam waist for each configuration if needed
    sim_data["sources"]['source1']["extra_args"]["width"] = float(beam_waist)
    
    output_path = f"{savepath_dir}/config_{i}"
    sim_data["output"]["savepath"]["path"] = output_path + "/" 
    # Create the output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)
    

    run_simulation(
        data=sim_data,
        source_freq=freq,
        res=res,
        savepath_dir=output_path,
        runtime=runtime,
        beam_waist=beam_waist,
        **config)

Now let's compare the E and S time-avg maps of the different configurations and plot it on a single graph!

In [ ]:
for i, config in enumerate(forebaffle_configs):
    print(f"\n{'='*50}")
    print(f"Analysing simulation {i+1}/{len(forebaffle_configs)}")
    print(f"{'='*50}\n")
    
    # Create a copy of data for each simulation
    sim_data = json.loads(json.dumps(data))
    sim_data["simulation"]["name"] += f"_config_{i}"
    
    savepath = f"{savepath_dir}/config_{i}"

    
    # For plotting the E and S time avg fields
    import numpy as np
    import os
    import matplotlib.pyplot as plt

    def load_fields(basepath, filename):
        # Construct the full path to the file
        filepath = os.path.join(basepath, filename)
        # Load the fields stored in npz files
        data = np.load(filepath)

        return data

    basepath = os.path.join(savepath)
    e_filename = 'efield_timeavg.npz'
    h_filename = 'hfield_timeavg.npz'
    xyzw_filename = 'xyzw.npz'
    e_data = load_fields(basepath, e_filename)
    h_data = load_fields(basepath, h_filename)
    xyzw_data = load_fields(basepath, xyzw_filename)
    print("E-field data keys:", e_data.files)
    print("H-field data keys:", h_data.files)
    print("XYZW data keys:", xyzw_data.files)


    # TE component (Ez, Hx, Hy)
    ez = e_data['ez_real'] + 1j * e_data['ez_imag']
    hx = h_data['hx_real'] + 1j * h_data['hx_imag']
    hy = h_data['hy_real'] + 1j * h_data['hy_imag']

    # S vector components for TE mode
    sx = -ez * np.conj(hy)
    sy = ez * np.conj(hx)
    sx_mag = np.abs(sx)
    sy_mag = np.abs(sy)
    s_total = np.sqrt(sx_mag**2 + sy_mag**2)
    s_total_db = 10 * np.log10(s_total / np.max(s_total) + 1e-20)  # in dB
    # efield magnitude
    ez_power = np.abs(ez)**2
    ez_power_db = 10 * np.log10(ez_power / np.max(ez_power) + 1e-20)  # in dB

    def plot_field(simname, field_db, title, filename, xcoords, ycoords, freq,
                vmin=-40, vmax=0, 
                savepath= os.path.join('./../processed_data/'),
                    show_plots=True,
                    mark_x = None):
        import matplotlib.pyplot as plt
        plt.style.use('default')
        
        plt.figure(figsize=(8, 6))
        plt.imshow(field_db.T, extent=(xcoords[0], xcoords[-1], ycoords[0], ycoords[-1]),
                origin='lower', cmap='inferno', vmin=vmin, vmax=vmax)
        if mark_x is not None:
            plt.axvline(x=mark_x, color='white', linestyle='--')
        plt.colorbar(label='dB')
        plt.title(title)
        plt.xlabel('x (mm)')
        plt.ylabel('y (mm)')

        if savepath:
            # Create directory with simname and frequency subdirectories
            save_dir = os.path.join(savepath, simname, f'{freq}GHz')
            os.makedirs(save_dir, exist_ok=True)
            plt.savefig(os.path.join(save_dir, filename), dpi=300)
            # Save as a svg file as well for publication
            plt.savefig(os.path.join(save_dir, filename).replace('.png', '.svg'), dpi=300) 
            print(f"{title} plot saved to: {os.path.join(save_dir, filename)}")
        if show_plots:
            plt.show()

    mark_x = -sim_data["simulation"]["primary_params"]["cell_size"]["x"]/2 + 5
    # E-field plot
    plot_field(simname = 'simple_single_lens_AddedComplexities_TRV_b_config_{}'.format(i), 
                field_db=ez_power_db, 
                title='E-field Magnitude (dB)', 
                filename='ez_magnitude_db.png', 
                xcoords=xyzw_data['x_coords'], 
                ycoords=xyzw_data['y_coords'],
                mark_x = mark_x,
                freq=freq)
    # S-vector plot
    plot_field(simname = 'simple_single_lens_AddedComplexities_TRV_b_config_{}'.format(i), 
                field_db= s_total_db, 
                title='Poynting Vector Magnitude (dB', 
                filename='s_magnitude_db.png', 
                xcoords=xyzw_data['x_coords'], 
                ycoords=xyzw_data['y_coords'],
                freq=freq)


Now let's compare the far field profiles of the different configuration and plot it on a single graph!

In [ ]:
import meepsat.field_analysis as analysis
import meepsat.helpers as helpers
import matplotlib.pyplot as plt
from numpy import power
plt.style.use('default')
plt.figure(figsize=(12, 6))

label_fb_config = ['spline', 'curved', 'linear']
far_field_results = {}

alpha= 0.7
shift_aperture_slice = True
for i, config in enumerate(forebaffle_configs):
    print(f"\n{'='*50}")
    print(f"Analysing simulation {i+1}/{len(forebaffle_configs)}")
    print(f"{'='*50}\n")
    
    savepath = f"{savepath_dir}/config_{i}"
    
    # For plotting the E and S time avg fields
    import numpy as np
    import os
    import matplotlib.pyplot as plt

    def load_fields(basepath, filename):
        # Construct the full path to the file
        filepath = os.path.join(basepath, filename)
        # Load the fields stored in npz files
        data = np.load(filepath)

        return data

    data = helpers.read_json(savepath + '/simple_single_lens_AddedComplexities_TRV_b_config_{}_simulation_data.json'.format(i))

    # print(data["sources"]["source1"]["frequecy"])
    # Define aperture radius OUTSIDE the loop
    aperture_radius = 46 #data["simulation"]["primary_params"]["cell_size"]["y"]/2 #data["apertures"]["aperture1"]["diameter"] / 2 #data["simulation"]["primary_params"]["cell_size"]["y"]/2 #data["apertures"]["aperture1"]["diameter"] / 2
    x_mm = -data["simulation"]["primary_params"]["cell_size"]["x"]/2 + 8.67  #data["apertures"]["aperture1"]["pos_x"]
    print(f"Aperture radius: {aperture_radius} mm, x position: {x_mm} mm")
    freq = data["sources"]["source1"]["frequecy"]
    zeropad = 15

    c = 2.998e+11  # Speed of light in mm/s
    wvl_meep = 1/freq
    real_freq = c / (wvl_meep * 1e9) #GHz
    print(f"Frequency to analyse: {real_freq} GHz, corresponding wavelength: {wvl_meep:.2f} mm")
    

    basepath = os.path.join(savepath)
    e_filename = 'efield_timeavg.npz'
    h_filename = 'hfield_timeavg.npz'
    xyzw_filename = 'xyzw.npz'
    e_data = load_fields(basepath, e_filename)
    h_data = load_fields(basepath, h_filename)
    xyzw_data = load_fields(basepath, xyzw_filename)
    print("E-field data keys:", e_data.files)
    print("H-field data keys:", h_data.files)
    print("XYZW data keys:", xyzw_data.files)


    # TE component (Ez, Hx, Hy)
    ez = e_data['ez_real'] + 1j * e_data['ez_imag']
    hx = h_data['hx_real'] + 1j * h_data['hx_imag']
    hy = h_data['hy_real'] + 1j * h_data['hy_imag']

    # S vector components for TE mode
    sx = -ez * np.conj(hy)
    sy = ez * np.conj(hx)
    sx_mag = np.abs(sx)
    sy_mag = np.abs(sy)
    s_total = np.sqrt(sx_mag**2 + sy_mag**2)
    s_total_db = 10 * np.log10(s_total / np.max(s_total) + 1e-20)  # in dB
    # efield magnitude
    ez_power = np.abs(ez)**2
    ez_power_db = 10 * np.log10(ez_power / np.max(ez_power) + 1e-20)  # in dB
    

    # ================== For Far Field Calculations ==================
    # MeepSAT (Ez is the dominant component for TE mode in MeepSAT simulations)
    x_index = (np.abs(xyzw_data['x_coords'] - (x_mm))).argmin()
    aperture_slice_meepsat = ez[x_index, :]  # Extract 1D slice at x=-60
    aperture_slice_meepsat = np.abs(aperture_slice_meepsat)

    # Set everything else to NaN except the values within the aperture diameter
    aperture_slice_meepsat = np.where(np.abs(xyzw_data['y_coords']) <= aperture_radius, aperture_slice_meepsat, np.nan)
    # aperture_slice_meepsat = np.abs(ez[x_index, :])
    # Remove NaN indices from both the power and the corresponding y-coordinates
    valid_indices = ~np.isnan(aperture_slice_meepsat)
    aperture_slice_meepsat = aperture_slice_meepsat[valid_indices]
    y_coords_meepsat = xyzw_data['y_coords'][valid_indices]
    aperture_slice_meepsat = aperture_slice_meepsat/np.max(aperture_slice_meepsat)  # Normalize the power for better comparison
    aperture_slice_meepsat_dB = 10 * np.log10(aperture_slice_meepsat + 1e-20)  # in dB

    # # # Plot the aperture slice for verification
    # plt.plot(y_coords_meepsat, aperture_slice_meepsat_dB, label=f"Aperture Slice - {label_fb_config[i]}")
    # plt.xlabel('Y-coordinate (mm)')
    # plt.ylabel('Normalized Power (dB)')
    # plt.title(f"Aperture Slice at x={x_mm} mm - {label_fb_config[i]}")
    # # plt.xlim(-aperture_radius, aperture_radius)
    # plt.legend()
    # plt.grid()

    meepsat_farfield_dict = analysis.meepsat_farfield(efield= aperture_slice_meepsat,
                coords= y_coords_meepsat,
                wavelength= wvl_meep,
                resolution= int(data['simulation']['primary_params']['resolution']),
                zero_pad_beam=zeropad,
                plot_label='MEEPSAT FDTD')
    
    far_field_results[label_fb_config[i]] = meepsat_farfield_dict


    # For finding the peaks
    from scipy.signal import find_peaks

    angle = meepsat_farfield_dict['angle']
    power_db = meepsat_farfield_dict['power_dB']
    
    
    
    # # Apply a moving average filter to smooth the power_db data
    # window_size = 50  # Adjust the window size as needed
    # power_db_smooth = np.convolve(power_db, np.ones(window_size)/window_size, mode='same')

    # # Find peaks
    # peak_idx, properties = find_peaks(
    #     power_db,
    #     height=-120,      # ignore very low sidelobes
    #     prominence=3     # adjust as needed
    # )
    
    # # Find peaks on the smoothed data
    # peak_idx, properties = find_peaks(
    #     power_db_smooth,
    #     height=-60,      # ignore very low sidelobes
    #     prominence=3     # adjust as needed
    # )
    
    
    # Find local maxima only
    peaks, _ = find_peaks(
        power_db,
        prominence=3, 
        distance=3
    )

    peak_angles = angle[peaks]
    peak_power = power_db[peaks]

    plt.plot(peak_angles, peak_power, label=f"Peaks - {label_fb_config[i]}")
    
    # from scipy.interpolate import PchipInterpolator

    # interp = PchipInterpolator(peak_angles, peak_power)

    # angle_smooth = np.linspace(
    #     peak_angles.min(),
    #     peak_angles.max(),
    #     1000
    # )

    # envelope = interp(angle_smooth)
    
    # plt.plot(angle, power, alpha=0.2)
    # plt.plot(angle_smooth, envelope, linewidth=2, label= label_fb_config[i])
    
    # Plot full pattern
    # plt.plot(angle, power_db, label=label_fb_config[i], alpha=alpha)
#     # # Plot peak points only
#     # plt.scatter(
#     #     angle[peak_idx],
#     #     power_db[peak_idx],
#     #     zorder=5,
#     #     label=f"Peaks - {label_fb_config[i]}"
#     # )
    
    # # Plot the peak 
    # plt.plot(
    #     angle[peak_idx],
    #     power_db[peak_idx],
    #     label=f"Peaks - {label_fb_config[i]}"
    # )

plt.xlabel('Angle (degrees)')
plt.ylabel('Power (dB)')
plt.xlim(0, 87)
# plt.ylim(-80,0)
plt.legend()

# # Save the figure
# save_dir = os.path.join('./../processed_data', 'simple_single_lens_AddedComplexities_TRV_b/noflairs', f'{freq}GHz')
# os.makedirs(save_dir, exist_ok=True)

# # # Peak
# # plt.savefig(os.path.join(save_dir, 'farfield_peaks_comparison.png'), dpi=300, bbox_inches='tight')
# # # Save as a svg file for better quality in publications
# # plt.savefig(os.path.join(save_dir, 'farfield_peaks_comparison.svg'), dpi=300, bbox_inches='tight')

# # # Profiles
# # plt.savefig(os.path.join(save_dir, 'farfield_comparison.png'), dpi=300, bbox_inches='tight')
# # # Save as a svg file for better quality in publications
# # plt.savefig(os.path.join(save_dir, 'farfield_comparison.svg'), dpi=300, bbox_inches='tight')

plt.show()

# Save the far field results as a dictionary for later analysis
save_dir = os.path.join('./../processed_data', 'simple_single_lens_AddedComplexities_TRV_b_noflairs', f'{freq}GHz')
os.makedirs(save_dir, exist_ok=True)
far_field_results_path = os.path.join(save_dir, 'far_field_results.npz')
# Save the dictionary as a numpy compressed file
np.savez_compressed(far_field_results_path, **far_field_results)

# #print range of y coords and delta y
# print(f"Y-coordinates range: {y_coords_meepsat.min()} mm to {y_coords_meepsat.max()} mm, Delta y: {y_coords_meepsat[1] - y_coords_meepsat[0]} mm")

# # Load the saved far field results for verification
# loaded_far_field_results = np.load(far_field_results_path, allow_pickle=True)
# print("Loaded far field results keys:", loaded_far_field_results.files)
